In [1]:
try:
    from databricks.sdk import WorkspaceClient
except ImportError:
    !pip install databricks-sdk
    from databricks.sdk import WorkspaceClient

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 822.0/822.0 kB 13.8 MB/s eta 0:00:00


In [3]:
# Importing Libraries
import json
import requests
import pandas as pd
from databricks.sdk import WorkspaceClient
from google.colab import userdata

# Defining functions

In [14]:
# ----- # ----- # ----- Get Jobs for Amazon ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def jobs_amazon(result_limit=100, offset=0, country=None):
    response = ''
    try:
        url = 'https://www.amazon.jobs/en/search.json'
        params = {
            'offset': offset,
            'result_limit': result_limit,
            'sort': 'relevant'
        }

        if country:
          params['normalized_country_code[]'] = country
        else:
          params['facets[]'] = 'normalized_country_code'

        headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'application/json, text/plain, */*',
        'Referer': 'https://www.amazon.jobs/en/search',
        'Accept-Encoding': 'gzip, deflate'
        }

        response = requests.get(url, params=params, headers=headers)
        response_json = json.loads(response.text)
        if 'jobs' in response_json:
          if len(response_json['jobs']) > 0:
            return True, response_json
          else:
            return False, response_json
        return False, {'Error': response.text}
    except Exception as e:
        print(str(e))
        print(f"Unable to connect to Amazon: {response.text}")
        return False, {'Error': response.text}

# ----- # ----- # ----- Fetch Job List ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def fetch_jobs():
    limit = 100
    offset = 0
    job_df_list = []

    # Perform Initial API Call
    job_bool, job_json = jobs_amazon(limit, offset)
    if job_bool:
      countries = {key for d in job_json['facets']['normalized_country_code_facet'] for key in d.keys()}

    # Start the loop
    for each_country in countries:
      limit = 100
      offset = 0
      while True:
          if offset%100 == 0:
            print(f'Offset: {offset}, Count: {limit + offset}, Country: {each_country}')
          job_bool, job_json = jobs_amazon(limit, offset, each_country)
          if job_bool:
            job_df_list.extend(job_json['jobs'])
            offset += limit
          else:
            break

    # Concatenate the dataset and clean
    jobs = pd.DataFrame(job_df_list)
    jobs.to_parquet('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/amazon_jobs.parquet', index=False)
    print(f'Total Jobs: {len(jobs)}')
    return jobs

In [13]:
# ----- # ----- # ----- Upload file to Databricks ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def upload_data(data_file):
  w = WorkspaceClient(
      host = userdata.get('DATABRICKS_HOST'),
      token = userdata.get('DATABRICKS_TOKEN')
  )

  volume_path = "/Volumes/workspace/jobs_raw/job_files/" + data_file.split('/')[-1]
  with open(data_file, "rb") as f:
      w.files.upload(volume_path, f)

  print(f"Successfully uploaded {data_file} to {volume_path}")

# New Architecture

In [15]:
job_frame = fetch_jobs()
upload_data('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/amazon_jobs.parquet')

Offset: 0, Count: 100, Country: ISR
Offset: 100, Count: 200, Country: ISR
Offset: 200, Count: 300, Country: ISR
Offset: 0, Count: 100, Country: CHN
Offset: 100, Count: 200, Country: CHN
Offset: 200, Count: 300, Country: CHN
Offset: 300, Count: 400, Country: CHN
Offset: 0, Count: 100, Country: PHL
Offset: 100, Count: 200, Country: PHL
Offset: 0, Count: 100, Country: COL
Offset: 100, Count: 200, Country: COL
Offset: 0, Count: 100, Country: SVK
Offset: 100, Count: 200, Country: SVK
Offset: 0, Count: 100, Country: PRT
Offset: 100, Count: 200, Country: PRT
Offset: 0, Count: 100, Country: SWE
Offset: 100, Count: 200, Country: SWE
Offset: 0, Count: 100, Country: SAU
Offset: 100, Count: 200, Country: SAU
Offset: 0, Count: 100, Country: GBR
Offset: 100, Count: 200, Country: GBR
Offset: 200, Count: 300, Country: GBR
Offset: 300, Count: 400, Country: GBR
Offset: 400, Count: 500, Country: GBR
Offset: 500, Count: 600, Country: GBR
Offset: 0, Count: 100, Country: AUT
Offset: 100, Count: 200, Country